# NB1 — Raw Agreement Analysis

In [ ]:
import os

# Mount Google Drive
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

import pandas as pd
import numpy as np

# Make sure output folders exist
os.makedirs("outputs/tables", exist_ok=True)
os.makedirs("outputs/graphs", exist_ok=True)

MASTER_PATH = "Master_Eval_Sheet.xlsx"

# ── Column names ────────────────────────────────────────────────
# Master sheet structure:
#   Row 1 = section labels  (Gold Reference + Annotation, H1-Ashish ...)
#   Row 2 = column names    (#, Sentence ID, ..., C1,C2,C3,C4,Total repeating)
#   Row 3 onwards = data
# We skip both header rows and assign unambiguous column names directly.

col_names = [
    "#", "Sentence_ID", "Source_File", "Source_Sentence", "Gold_Category",
    "LLM", "Has_Errors", "Error_Span", "Annotated_Category", "Description",
    "Corrected_Sentence",
    "H1_C1", "H1_C2", "H1_C3", "H1_C4", "H1_Total",
    "H2_C1", "H2_C2", "H2_C3", "H2_C4", "H2_Total",
    "L1_C1", "L1_C2", "L1_C3", "L1_C4", "L1_Total",
    "L2_C1", "L2_C2", "L2_C3", "L2_C4", "L2_Total",
    "L3_C1", "L3_C2", "L3_C3", "L3_C4", "L3_Total",
    "L4_C1", "L4_C2", "L4_C3", "L4_C4", "L4_Total",
]

df = pd.read_excel(
    MASTER_PATH,
    sheet_name="Master Eval Sheet",
    header=None,
    skiprows=2,
    names=col_names
)

# Drop empty trailing rows and the summary TOTAL row
df = df[df["#"].notna()].copy()
df = df[df["#"].astype(str) != "TOTAL"].copy()
df = df.reset_index(drop=True)

print("Loaded:", df.shape)
print("Columns:", df.columns.tolist())
print("\nSample:")
print(df[["Sentence_ID", "Gold_Category", "LLM", "H1_C1", "H2_C1", "L1_C1", "L2_C1", "L3_C1", "L4_C1"]].head(3))

# ── Rater groups ────────────────────────────────────────────────
G1_HUMANS           = ["H1", "H2"]
G2_ANNOTATOR_LLMS   = ["L1", "L2"]
G3_NON_ANNOTATOR    = ["L3", "L4"]
G4_ALL_LLMS         = ["L1", "L2", "L3", "L4"]
G5_HUMANS_ANN       = ["H1", "H2", "L1", "L2"]
G6_HUMANS_NONANN    = ["H1", "H2", "L3", "L4"]
G7_ALL              = ["H1", "H2", "L1", "L2", "L3", "L4"]

ALL_GROUPS = {
    "G1_Humans":             G1_HUMANS,
    "G2_Annotator_LLMs":     G2_ANNOTATOR_LLMS,
    "G3_NonAnnotator_LLMs":  G3_NON_ANNOTATOR,
    "G4_All_LLMs":           G4_ALL_LLMS,
    "G5_Humans+Ann_LLMs":    G5_HUMANS_ANN,
    "G6_Humans+NonAnn_LLMs": G6_HUMANS_NONANN,
    "G7_All":                G7_ALL,
}

CATEGORIES = ["Script Normalization", "Spelling & Typographical Error",
              "Grammatical Error", "Code-Mixing / Wrong Language",
              "Correct Sentence / No Errors"]

TASKS = ["C1", "C2", "C3", "C4"]

print("\nRater groups and tasks defined.")


In [ ]:
# ── Cell 2: Helper functions for raw agreement ────────────────

def get_scores(data, raters, task):
    # Returns a 2D list: rows = items, cols = raters
    cols = [r + "_" + task for r in raters]
    return data[cols].values.tolist()


def exact_agreement_pct(scores_2d):
    # % of rows where all raters gave the exact same score
    total = len(scores_2d)
    agreed = 0
    for row in scores_2d:
        if len(set(row)) == 1:
            agreed += 1
    return round((agreed / total) * 100, 2) if total > 0 else None


def adjacent_agreement_pct(scores_2d):
    # % of rows where all raters are within 1 point of each other
    # Only meaningful for ordinal tasks C2 and C4
    total = len(scores_2d)
    agreed = 0
    for row in scores_2d:
        if max(row) - min(row) <= 1:
            agreed += 1
    return round((agreed / total) * 100, 2) if total > 0 else None


print("Helper functions ready.")


In [ ]:
# ── Cell 3: Overall raw agreement (all 100 rows) ──────────────

overall_results = []

for group_name, raters in ALL_GROUPS.items():
    for task in TASKS:
        scores = get_scores(df, raters, task)

        exact = exact_agreement_pct(scores)

        # Adjacent % only for ordinal tasks
        if task in ["C2", "C4"]:
            adjacent = adjacent_agreement_pct(scores)
        else:
            adjacent = None

        overall_results.append({
            "Group": group_name,
            "Task": task,
            "N_Raters": len(raters),
            "N_Items": len(scores),
            "Exact_%": exact,
            "Adjacent_%": adjacent,
        })

overall_df = pd.DataFrame(overall_results)
print("Overall raw agreement:")
print(overall_df.to_string(index=False))


In [ ]:
# ── Cell 4: Category-wise raw agreement ───────────────────────

category_results = []

for category in CATEGORIES:
    cat_df = df[df["Gold_Category"] == category].copy()

    for group_name, raters in ALL_GROUPS.items():
        for task in TASKS:
            scores = get_scores(cat_df, raters, task)

            exact = exact_agreement_pct(scores)

            if task in ["C2", "C4"]:
                adjacent = adjacent_agreement_pct(scores)
            else:
                adjacent = None

            category_results.append({
                "Category": category,
                "Group": group_name,
                "Task": task,
                "N_Raters": len(raters),
                "N_Items": len(scores),
                "Exact_%": exact,
                "Adjacent_%": adjacent,
            })

category_df = pd.DataFrame(category_results)
print("Category-wise raw agreement (first 20 rows):")
print(category_df.head(20).to_string(index=False))


In [ ]:
# ── Cell 5: Save results to Excel ─────────────────────────────

output_path = "outputs/tables/NB1_Raw_Agreement_Results.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    overall_df.to_excel(writer, sheet_name="Overall", index=False)
    category_df.to_excel(writer, sheet_name="By_Category", index=False)

print("Saved:", output_path)
print("Sheets: Overall,  By_Category")
